In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#------------------------------------------------------------
# Load LegacyClimate Reconstructions and extract samples
# for modern (0 - 200 yr BP) and 6 ka timeslices
#------------------------------------------------------------

base_dir = "..../LegacyClimate/LegacyClimate_csv_files/...." 
files = [
    "climate_reconstruction_asia.csv",
    "climate_reconstruction_europe.csv",
    "climate_reconstruction_north_america_east.csv",
    "climate_reconstruction_north_america_west.csv"
]

legacy_0k_df = pd.DataFrame()
legacy_6k_df = pd.DataFrame()

for file in files:
    region_df_full = pd.read_csv(base_dir + file)
    region_siteid = list(set(region_df_full['ID (Site)'].values))
    region_df = pd.DataFrame()
    for id in region_siteid:
        site_df = region_df_full[region_df_full["ID (Site)"] == id]
        med_age = site_df["Age [ka BP] (median)"].values
        cnt_0 = len([i for i in med_age if 0 <= i <= 0.2])
        cnt_6 = len([i for i in med_age if 5.8 <= i <= 6.2])
        if (cnt_0 > 0) & (cnt_6 > 0):
            region_df = pd.concat([region_df, site_df], ignore_index=True, axis=0)

    age_uncertainty = region_df['Age max [ka]'] - region_df["Age min [ka]"]
    indices_6k = []
    indices_0k = []

    for i in range(len(region_df)):
        if (5.8 <= region_df["Age [ka BP] (median)"][i] <= 6.2):
            indices_6k.append(i)

        elif (-0.2 <= region_df["Age [ka BP] (median)"][i] <= 0.2):
            indices_0k.append(i)

        else: continue

    region_0k_df = region_df.iloc[indices_0k]
    legacy_0k_df = pd.concat([legacy_0k_df, region_0k_df], ignore_index=True, axis=0)

    region_6k_df = region_df.iloc[indices_6k]
    legacy_6k_df = pd.concat([legacy_6k_df, region_6k_df], ignore_index=True, axis=0)

legacy_0k_df.to_excel("climate_reconstruction_0k.xlsx", index=False)
legacy_6k_df.to_excel("climate_reconstruction_6k.xlsx", index=False)

In [ ]:
#------------------------------------------------------------
# Compile a compilation of reconstruction anomalies 
#------------------------------------------------------------

clim_recon_0k = pd.read_excel("climate_reconstruction_0k.xlsx")
clim_recon_0k.drop(list(clim_recon_0k.filter(regex = "Dissimilarity|Reference|Depth")), axis=1, inplace=True)

clim_recon_6k = pd.read_excel("climate_reconstruction_6k.xlsx")
clim_recon_6k.drop(list(clim_recon_6k.filter(regex = "Dissimilarity|Reference|Depth")), axis=1, inplace=True)

anom_df_full = pd.DataFrame()


site_ids = list(set(clim_recon_0k["ID (Site)"]))

for id in site_ids:

    anom_site_df = pd.DataFrame()

    # Extract 0k and 6k data from the site
    site_0k = clim_recon_0k[clim_recon_0k["ID (Site)"] == id]
    site_6k = clim_recon_6k[clim_recon_6k["ID (Site)"] == id]
    
    # Write site metadata to site df
    anom_site_df.at[0, "Event"] = site_0k["Event"].values[0]
    anom_site_df.at[0, "ID (Dataset)"] = site_0k["ID (Dataset)"].values[0]
    anom_site_df.at[0, "Continent"] = site_0k["Continent"].values[0]
    anom_site_df.at[0, "ID (Site)"] = site_0k["ID (Site)"].values[0]
    anom_site_df.at[0, "Site"] = site_0k["Site"].values[0]
    anom_site_df.at[0, "Latitude"] = site_0k["Latitude"].values[0]
    anom_site_df.at[0, "Longitude"] = site_0k["Longitude"].values[0]

    # Iterate through all the reconstruction columns
    for i in np.arange(11, 36, 2):
        # Calculate the mean reconstruction and mean reconstruction error from the 0k and 6k samples
        mean_var_0k = np.mean(site_0k.values[:, i])
        mean_err_0k = np.mean(site_0k.values[:, i + 1])

        mean_var_6k = np.mean(site_6k.values[:, i])
        mean_err_6k = np.mean(site_6k.values[:, i + 1])

        # Assume that the means will form an anomaly distribution with mean as the diff. of means and std
        # as the sum of std's
        # Draw 10000 samples from that anomaly distribution
        anom_samp = np.random.normal(mean_var_6k - mean_var_0k, mean_err_6k + mean_err_0k, 10000)

        # Calcuclate mean anomaly and mean error
        mean_anom = np.mean(anom_samp)
        err_anom = np.std(anom_samp)

        # Add this data to the site df
        anom_site_df.at[0, site_0k.columns[i]] = mean_anom
        anom_site_df.at[0, site_0k.columns[i + 1]] = err_anom

    # Attach site df to full anomaly df
    anom_df_full = pd.concat([anom_df_full, anom_site_df], axis=0, ignore_index=True)

# Write full anomaly df to excel file
anom_df_full.to_excel("../Data/Proxies/LC_climate_reconstruction_anomalies.xlsx", index=False)
